# Parcels environment : base

In [1]:
import parcels
import numpy as np
from datetime import timedelta
from glob import glob
import matplotlib.pyplot as plt
import xarray as xr
import os

def u2rho_2d (var_u):
    [Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[:,1:-1]=0.5*(var_u[:,1:]+var_u[:,:-1])
    var_rho[:,0]=var_rho[:,1]
    var_rho[:,-1]=var_rho[:,-2]
    return var_rho
    
def v2rho_2d (var_v):
    [M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((Mp,Lp))
    var_rho[1:-1,:]=0.5*(var_v[1:,:]+var_v[:-1,:])
    var_rho[0,:]=var_rho[1,:]
    var_rho[-1,:]=var_rho[-2,:]
    return var_rho

def u2rho_3d (var_u):
    [N,Mp,L]=var_u.shape
    Lp=L+1
    Lm=L-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,:,1:-1]=0.5*(var_u[:,:,1:]+var_u[:,:,:-1])
    var_rho[:,:,0]=var_rho[:,:,1]
    var_rho[:,:,-1]=var_rho[:,:,-2]
    return var_rho
    
def v2rho_3d (var_v):
    [N,M,Lp]=var_v.shape
    Mp=M+1
    Mm=M-1
    var_rho=np.zeros((N,Mp,Lp))
    var_rho[:,1:-1,:]=0.5*(var_v[:,1:,:]+var_v[:,:-1,:])
    var_rho[:,0,:]=var_rho[:,1,:]
    var_rho[:,-1,:]=var_rho[:,-2,:]
    return var_rho

def spheric_dist(lat1, lat2, lon1, lon2):
    """
    Compute the spherical distance between two points on Earth.
    
    Parameters:
    lat1, lat2 : array-like
        Latitude of the two points (in degrees).
    lon1, lon2 : array-like
        Longitude of the two points (in degrees).
    
    Returns:
    dist : array-like
        The spherical distance between the points (in meters).
    """
    
    # Earth radius in meters
    R = 6367442.76
    
    # Determine proper longitudinal shift
    l = np.abs(lon2 - lon1)
    l[l >= 180] = 360 - l[l >= 180]
    
    # Convert decimal degrees to radians
    deg2rad = np.pi / 180
    lat1 = lat1 * deg2rad
    lat2 = lat2 * deg2rad
    l = l * deg2rad
    
    # Compute the distances
    dist = R * np.arcsin(np.sqrt(((np.sin(l) * np.cos(lat2)) ** 2) + 
                                 ((np.sin(lat2) * np.cos(lat1)) - 
                                  (np.sin(lat1) * np.cos(lat2) * np.cos(l))) ** 2))
    
    return dist

def spheric_dist_one(lat1, lat2, lon1, lon2):
    """计算两个经纬度点之间的球面距离（标量版）"""
    # 处理经度差
    l = np.abs(lon2 - lon1)
    if l >= 180:
        l = 360 - l
    
    # 转换为弧度
    deg2rad = np.pi / 180
    lat1_rad = lat1 * deg2rad
    lat2_rad = lat2 * deg2rad
    l_rad = l * deg2rad
    
    # 球面距离公式
    distance = 6371 * np.arccos(
        np.sin(lat1_rad) * np.sin(lat2_rad) + 
        np.cos(lat1_rad) * np.cos(lat2_rad) * np.cos(l_rad)
    )
    return distance
def transunit_spher2flat(u,v,lat):
    v1=v*1852*60
    u1=u*1852*60*np.cos(lat*np.pi/180)
    return u1,v1

def trans_vel_roms(u,v,angle):
    # np.cos(angle*np.pi/180)
    # np.sin(angle*np.pi/180)
    u_east=u*np.cos(angle*np.pi/180)-v*np.sin(angle*np.pi/180)
    v_north=u*np.sin(angle*np.pi/180)+v*np.cos(angle*np.pi/180)
    return u_east,v_north

from scipy.interpolate import griddata

def griddata_matlab(lona,lata,ugos,lon_rho,lat_rho):
    points = np.column_stack((lona.ravel(), lata.ravel()))
    values = ugos.ravel()
    
    # 步骤2: 创建目标网格点
    target_points = np.column_stack((lon_rho.ravel(), lat_rho.ravel()))
    
    # 步骤3: 执行插值
    ugos_interp = griddata(points, values, target_points, method='linear')
    
    # 步骤4: 重塑为原始网格形状
    ugos_interp = ugos_interp.reshape(lon_rho.shape)
    return ugos_interp

def snake_scan(matrix):
    """
    ravel() for particles
    matrix: 
        
    flat_array: 1d arrays
    """
    rows, cols = matrix.shape
    result = []
    
    for i in range(rows):
        if i % 2 == 0:  # 偶数行：从左到右
            result.extend(matrix[i, :])
        else:           # 奇数行：从右到左
            result.extend(matrix[i, ::-1])
    
    return np.array(result)

import numpy as np
from scipy.interpolate import griddata

def particle_roughdistr_onetime(lonp, latp, cI, cJ, NP, Range, shape='circle'):
    '''    
    para:
        lonp, latp: grid
        cI, cJ: center of region
        NP: numbers of p
        Range: sub region
        shape:  ('box' or 'circle')
    '''
    squareP = int(np.sqrt(NP))
    Len = int(Range)
    
    # 
    if Len % 2 == 0:
        II = [int(cI - Len/2), int(cI + Len/2)]
        JJ = [int(cJ - Len/2), int(cJ + Len/2)]
    else:
        II = [int(cI - (Len+1)/2), int(cI + Len/2)]
        JJ = [int(cJ - (Len+1)/2), int(cJ + Len/2)]
    
    # sub region
    lonp1 = lonp[JJ[0]:JJ[1], II[0]:II[1]]
    latp1 = latp[JJ[0]:JJ[1], II[0]:II[1]]
    
    # 
    Iin, Jin = np.meshgrid(np.arange(lonp1.shape[0]), 
                          np.arange(lonp1.shape[1]))
    
    points = np.column_stack((Iin.ravel(), Jin.ravel()))

    # choose shape
    if shape == 'box':
        # 
        I1 = np.linspace(0, lonp1.shape[0]-1, squareP)
        J1 = np.linspace(0, lonp1.shape[1]-1, squareP)
        I11, J11 = np.meshgrid(I1, J1)
        
        # 
        lonp2 = griddata(points, lonp1.ravel(), (I11, J11), method='linear')
        latp2 = griddata(points, latp1.ravel(), (I11, J11), method='linear')
        return snake_scan(lonp2), snake_scan(latp2)
    
    elif shape == 'circle':

        center_x = lonp1.shape[0] / 2
        center_y = lonp1.shape[1] / 2
        
        radius = min(lonp1.shape[0], lonp1.shape[1]) / 2
        
        r = radius * np.sqrt(np.random.rand(NP))  #
        theta = 2 * np.pi * np.random.rand(NP)
        
        # 
        x = center_x + r * np.cos(theta)
        y = center_y + r * np.sin(theta)
        
        # 
        points_target = np.column_stack((x, y))
        lonp2 = griddata(points, lonp1.ravel(), points_target, method='linear')
        latp2 = griddata(points, latp1.ravel(), points_target, method='linear')
        
        return lonp2, latp2



def generate_release_times_hourly(total_particles, particles_per_hour, start_time=0):
    """
    :
        total_particles: 
        particles_per_hour: 
        start_time: 
    :
        timep2: 
    """
    total_particles = int(total_particles)
    particles_per_hour = int(particles_per_hour)

    hours_needed = int(np.ceil(total_particles / particles_per_hour))

    timep2 = []
    for hour in range(hours_needed):
        particles_this_hour = min(particles_per_hour, total_particles - len(timep2))
        hour_start = start_time + hour * 3600
        timep2.extend([hour_start] * particles_this_hour)
    
    return np.array(timep2)

In [2]:
grid_dir='/meddy/simingzhang/Data/RB_iceland_data/'
wave_dir='/meddy/simingzhang/Data/Parcels_data/'
# nowave_dir='/meddy/simingzhang/Data/RB_iceland_data/iceland_no_wave/'
grdname='niskin2km_500m_grd.nc'
# hisname_w='z_niskin2km_his_hf_depth_500m_grd.0002.nc'
# hisname_nw='z_niskin2km_his_smooth_depth_500m_grd.0002.nc'

grdname=f'{grid_dir}{grdname}'
# wavename=f'{wave_dir}{hisname_w}'
# nowavename=f'{nowave_dir}{hisname_nw}'

# wavename=f'{wave_dir}wavecase_modified_cg.nc'
# nowavename=f'{wave_dir}nowavecase_modified_cg.nc'
wavename=f'{wave_dir}wavecase_modified_vel_cg_kaiser1_rot.nc'
nowavename=f'{wave_dir}nowavecase_modified_vel_cg_kaiser1_rot.nc'

In [3]:
grid=xr.open_dataset(grdname)
grid = grid.swap_dims({'eta_u': 'eta_rho','xi_v':'xi_rho'})

lon_rho=grid['lon_rho']
lat_rho=grid['lat_rho']
h=grid['h']
angle=grid['angle'].values
lonmin=np.min(lon_rho.values)
lonmax=np.max(lon_rho.values)
latmin=np.min(lat_rho.values)
latmax=np.max(lat_rho.values)
lonmin,lonmax,latmin,latmax
grid

<xarray.Dataset> Size: 19MB
Dimensions:    (one: 1, eta_rho: 287, xi_rho: 287, bath: 1, xi_u: 286,
                eta_v: 286, eta_psi: 286, xi_psi: 286)
Dimensions without coordinates: one, eta_rho, xi_rho, bath, xi_u, eta_v,
                                eta_psi, xi_psi
Data variables: (12/34)
    xl         (one) float64 8B ...
    el         (one) float64 8B ...
    depthmin   (one) float64 8B ...
    depthmax   (one) float64 8B ...
    spherical  (one) |S1 1B ...
    angle      (eta_rho, xi_rho) float64 659kB -10.14 -10.14 ... -11.33 -11.33
    ...         ...
    lat_v      (eta_v, xi_rho) float64 657kB ...
    lat_psi    (eta_psi, xi_psi) float64 654kB ...
    mask_rho   (eta_rho, xi_rho) float64 659kB ...
    mask_u     (eta_rho, xi_u) float64 657kB ...
    mask_v     (eta_v, xi_rho) float64 657kB ...
    mask_psi   (eta_psi, xi_psi) float64 654kB ...
Attributes:
    title:    Solomon Model
    date:     08-Apr-2018
    type:     ROMS grid file

In [4]:
ds = xr.open_dataset(wavename)
ds
# depth=ds['depth'].values

<xarray.Dataset> Size: 32GB
Dimensions:     (time: 2148, eta_rho: 287, xi_rho: 287, depth: 1, xi_u: 286,
                 eta_v: 286, depth_2: 1)
Coordinates:
  * time        (time) float64 17kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
    lon_rho     (eta_rho, xi_rho) float64 659kB ...
    lat_rho     (eta_rho, xi_rho) float64 659kB ...
  * depth       (depth) float32 4B -2.0
Dimensions without coordinates: eta_rho, xi_rho, xi_u, eta_v, depth_2
Data variables: (12/44)
    ocean_time  (time) float32 9kB ...
    u           (time, depth, eta_rho, xi_u) float32 705MB ...
    u_rho       (time, depth, eta_rho, xi_rho) float64 1GB ...
    v           (time, depth, eta_v, xi_rho) float32 705MB ...
    v_rho       (time, depth, eta_rho, xi_rho) float64 1GB ...
    Th1         (time, depth_2, eta_rho, xi_rho) float32 708MB ...
    ...          ...
    Th84        (time, depth_2, eta_rho, xi_rho) float32 708MB ...
    Th90        (time, depth_2, eta_rho, xi_rho) float32 708MB ...
    Th96        (time, depth_2, eta_rho, xi_rho) float32 708MB ...
    Th102       (time, depth_2, eta_rho, xi_rho) float32 708MB ...
    Th108       (time, depth_2, eta_rho, xi_rho) float32 708MB ...
    Th114       (time, depth_2, eta_rho, xi_rho) float32 708MB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.1 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Wed Oct 29 09:45:58 2025: cdo merge rot_helmholtz.0002.nc ....
    NCO:          netCDF Operators version 5.3.5 (Homepage = http://nco.sf.ne...
    CDO:          Climate Data Operators version 2.4.1 (https://mpimet.mpg.de...

In [5]:
# u1=u2rho_3d(np.squeeze(ds['u'].values))
# v1=v2rho_3d(np.squeeze(ds['v'].values))
# ueast,vnorth=trans_vel_roms(u1,v1,angle)
# ds['u_rho'].values[:,0,:,:]=ueast
# ds['v_rho'].values[:,0,:,:]=vnorth
# ds.to_netcdf(f'{wave_dir}wavecase_modified_cg_uni1.nc')

# Parcel
### 1. wave case

In [6]:
wavename
# filenames = {"U": wavename, "V": wavename,"PI2": wavename,"PI4": wavename,"PI6": wavename,"PI8": wavename,"PI10": wavename,
#             "PI12": wavename,"PI16": wavename,"PI20": wavename,"PI30": wavename,"PI50": wavename,"PI60": wavename,"PI100": wavename}

# filenames = {"U": wavename, "V": wavename}

# filenames = {"U": wavename, "V": wavename,"Th1": wavename,"Th2": wavename,"Th3": wavename,"Th4": wavename,"Th5": wavename,
#             "Th6": wavename,"Th7": wavename,"Th8": wavename,"Th9": wavename,"Th10": wavename,"Th11": wavename,"Th12": wavename,
#             "Th13": wavename,"Th14": wavename,"Th15": wavename,"Th16": wavename,"Th17": wavename,"Th18": wavename,"Th21": wavename,
#             "Th24": wavename,"Th27": wavename,"Th30": wavename,"Th33": wavename,"Th36": wavename,"Th39": wavename,"Th42": wavename,
#              "Th45": wavename,"Th48": wavename,"Th54": wavename,"Th60": wavename,"Th66": wavename,"Th72": wavename,"Th78": wavename,
#              "Th84": wavename,"Th90": wavename,"Th96": wavename,"Th102": wavename,"Th108": wavename,"Th114": wavename}

# 创建变量名列表
variable_names = ["U", "V"]
variable_names += [f"Th{i}" for i in range(1, 19)]
variable_names += [f"Th{i}" for i in [21, 24, 27, 30, 33, 36, 39, 42, 45, 48,
                                      54, 60, 66, 72, 78, 84, 90, 96, 102, 108, 114]]

# 创建字典
filenames = {var: wavename for var in variable_names}

# 打印结果
# print(filenames)
filenames

{'U': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'V': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th1': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th2': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th3': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th4': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th5': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th6': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th7': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th8': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th9': '/meddy/simingzhang/Data/Parcels_data/wavecase_modified_vel_cg_kaiser1_rot.nc',
 'Th10': '/meddy/simingzhang/Data/Pa

In [7]:

all_vars = ["U", "V"] + [f"Th{i}" for i in list(range(1, 19)) + [21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 54, 60, 66, 72, 78, 84, 90, 96, 102, 108, 114]]

# 创建 variables 字典
variables = {
    var: "u_rho" if var == "U" else "v_rho" if var == "V"  else var
    for var in all_vars
}

# 维度模板
dim_template = {
    "lat": "lat_rho",
    "lon": "lon_rho",
    "time": "ocean_time",
    "depth": "depth"
}

# 创建 dimensions 字典
dimensions = {var: dim_template.copy() for var in all_vars}

chunks=500
cs = {"time": ("ocean_time", 1), "lat": ("lat_rho", chunks), "lon": ("lon_rho", chunks),"depth": ("depth", chunks)}
# ,chunksize=cs
fieldset = parcels.FieldSet.from_netcdf(filenames, variables, dimensions,allow_time_extrapolation=True)
# ,allow_time_extrapolation=True
# 


In [8]:
fieldset

<FieldSet>
    fields:
        <Field>
            name            : 'U'
            grid            : CurvilinearZGrid(lon=array([[-30.93, -30.89, -30.86, ..., -21.72, -21.69, -21.65],
               [-30.92, -30.89, -30.85, ..., -21.71, -21.68, -21.65],
               [-30.91, -30.88, -30.85, ..., -21.71, -21.68, -21.64],
               ...,
               [-29.14, -29.11, -29.08, ..., -20.03, -20.00, -19.97],
               [-29.13, -29.10, -29.07, ..., -20.02, -19.99, -19.96],
               [-29.13, -29.09, -29.06, ..., -20.02, -19.99, -19.95]], dtype=float32), lat=array([[ 57.60,  57.59,  57.59, ...,  56.71,  56.71,  56.71],
               [ 57.61,  57.61,  57.61, ...,  56.73,  56.73,  56.72],
               [ 57.63,  57.63,  57.62, ...,  56.75,  56.74,  56.74],
               ...,
               [ 62.24,  62.23,  62.23, ...,  61.36,  61.36,  61.35],
               [ 62.25,  62.25,  62.25, ...,  61.37,  61.37,  61.37],
               [ 62.27,  62.26,  62.26, ...,  61.39,  61.39, 

In [9]:
# fieldset.U.grid.time *= 3600  # 秒 → 小时
# lonp1.shape

In [10]:
# fieldset.U.grid.time/1e9

In [11]:
def DeleteParticle(particle, fieldset, time):
    if (particle.lon < -30.92595646869225+1.5) or (particle.lon > -19.9541352315797-1.5) or \
       (particle.lat < 56.70597152564239+0.5) or (particle.lat > 62.267609501148414-0.5):
        particle.delete()
        

def SampleT(particle, fieldset, time):
    particle.ue = fieldset.U[time, particle.depth, particle.lat, particle.lon]
    particle.ve = fieldset.V[time, particle.depth, particle.lat, particle.lon]
    particle.th1 = fieldset.Th1[time, particle.depth, particle.lat, particle.lon]
    particle.th2 = fieldset.Th2[time, particle.depth, particle.lat, particle.lon]
    particle.th3 = fieldset.Th3[time, particle.depth, particle.lat, particle.lon]
    particle.th4 = fieldset.Th4[time, particle.depth, particle.lat, particle.lon]
    particle.th5 = fieldset.Th5[time, particle.depth, particle.lat, particle.lon]
    particle.th6 = fieldset.Th6[time, particle.depth, particle.lat, particle.lon]
    particle.th7 = fieldset.Th7[time, particle.depth, particle.lat, particle.lon]
    particle.th8 = fieldset.Th8[time, particle.depth, particle.lat, particle.lon]
    particle.th9 = fieldset.Th9[time, particle.depth, particle.lat, particle.lon]
    particle.th10 = fieldset.Th10[time, particle.depth, particle.lat, particle.lon]
    particle.th11 = fieldset.Th11[time, particle.depth, particle.lat, particle.lon]
    particle.th12 = fieldset.Th12[time, particle.depth, particle.lat, particle.lon]
    particle.th13 = fieldset.Th13[time, particle.depth, particle.lat, particle.lon]
    particle.th14 = fieldset.Th14[time, particle.depth, particle.lat, particle.lon]
    particle.th15 = fieldset.Th15[time, particle.depth, particle.lat, particle.lon]
    particle.th16 = fieldset.Th16[time, particle.depth, particle.lat, particle.lon]
    particle.th17 = fieldset.Th17[time, particle.depth, particle.lat, particle.lon]
    particle.th18 = fieldset.Th18[time, particle.depth, particle.lat, particle.lon]
    particle.th21 = fieldset.Th21[time, particle.depth, particle.lat, particle.lon]
    particle.th24 = fieldset.Th24[time, particle.depth, particle.lat, particle.lon]
    particle.th27 = fieldset.Th27[time, particle.depth, particle.lat, particle.lon]
    particle.th30 = fieldset.Th30[time, particle.depth, particle.lat, particle.lon]
    particle.th33 = fieldset.Th33[time, particle.depth, particle.lat, particle.lon]
    particle.th36 = fieldset.Th36[time, particle.depth, particle.lat, particle.lon]
    particle.th39 = fieldset.Th39[time, particle.depth, particle.lat, particle.lon]
    particle.th42 = fieldset.Th42[time, particle.depth, particle.lat, particle.lon]
    particle.th45 = fieldset.Th45[time, particle.depth, particle.lat, particle.lon]
    particle.th48 = fieldset.Th48[time, particle.depth, particle.lat, particle.lon]
    particle.th54 = fieldset.Th54[time, particle.depth, particle.lat, particle.lon]
    particle.th60 = fieldset.Th60[time, particle.depth, particle.lat, particle.lon]
    particle.th66 = fieldset.Th66[time, particle.depth, particle.lat, particle.lon]
    particle.th72 = fieldset.Th72[time, particle.depth, particle.lat, particle.lon]
    particle.th78 = fieldset.Th78[time, particle.depth, particle.lat, particle.lon]
    particle.th84 = fieldset.Th84[time, particle.depth, particle.lat, particle.lon]
    particle.th90 = fieldset.Th90[time, particle.depth, particle.lat, particle.lon]
    particle.th96 = fieldset.Th96[time, particle.depth, particle.lat, particle.lon]
    particle.th102 = fieldset.Th102[time, particle.depth, particle.lat, particle.lon]
    particle.th108 = fieldset.Th108[time, particle.depth, particle.lat, particle.lon]
    particle.th114 = fieldset.Th114[time, particle.depth, particle.lat, particle.lon]



def CheckOutOfBounds(particle, fieldset, time):
    if particle.state == StatusCode.ErrorOutOfBounds:
        particle.delete()
def CheckError(particle, fieldset, time):
    if particle.state >= 50:  # This captures all Errors
        particle.delete()

In [12]:
lon_rho.values.shape

(287, 287)

In [13]:
lonp=lon_rho.values
latp=lat_rho.values
lonp=lon_rho.values
latp=lat_rho.values
cI=143
cJ=143
# NP=17*17
# NP=25*25
NP=50*50
boxlen=70
lonp1,latp1=particle_roughdistr_onetime(lonp,latp,cI,cJ,NP,boxlen,'box')

during=89.5 # days
total_seconds = during * 86400 
# outputwavename=f'{wave_dir}wave_pars_P{latp1.shape[0]}T{during}days.zarr'
# lonp2= np.tile(lonp1, int(nrelease))
# latp2= np.tile(latp1, int(nrelease))
# timep2=np.repeat(np.arange(0, int(nrelease))*timedelta(hours=repeatdt).total_seconds(),npart)
# outputwavename=f'{wave_dir}wave_pars_P{int(lonp2.shape[0])}T{during}days.zarr'
outputwavename=f'{wave_dir}wave_pars_P{int(lonp1.shape[0])}T{during}days_rot.zarr'

# Initial setup

In [14]:
# plt.plot(lon_rho.values[0,:],lat_rho.values[0,:],color='r')
# plt.plot(lon_rho.values[-1,:],lat_rho.values[-1,:],color='r')
# plt.plot(lon_rho.values[:,0],lat_rho.values[:,0],color='r')
# plt.plot(lon_rho.values[:,-1],lat_rho.values[:,-1],color='r')

# plt.scatter(lonp1,latp1,s=0.05)
# plt.title(f'total num {lonp1.shape[0]}')
# plt.scatter(lonp1[0:80],latp1[0:80],s=5)
dist=spheric_dist_one(latp1[0], latp1[2], lonp1[0], lonp1[2])
# spheric_dist_one(latp[0,1], latp[1,1], lonp[0,1], lonp[1,1])

# ship speed 10 - 12 knots ~ 20 km /h
# NP=2500, 14 days
# NP=625 , 7 days
# NP=289, 5 days

In [15]:
dist=spheric_dist_one(latp1[0], latp1[1], lonp1[0], lonp1[1])
np_per_hour=np.ceil(20/dist)
dist,np.ceil(np_per_hour),np.ceil(NP/np.ceil(np_per_hour)/24),np.ceil(NP/np.ceil(np_per_hour))
#pair's distance, throw particle per hour, days to finish

(2.6497673772568953, 8.0, 14.0, 313.0)

In [16]:
timep2=generate_release_times_hourly(lonp1.shape[0], np_per_hour, start_time=0)
# timep2

In [ ]:


# late start
assert len(lonp1) == len(latp1), "经度和纬度数组长度不一致"

# repeatdt = timedelta(hours=6)
pset = parcels.ParticleSet.from_list(
    fieldset=fieldset,
    pclass=parcels.JITParticle.add_variable("ue").add_variable("ve").add_variable("th1").add_variable("th2").add_variable("th3").
    add_variable("th4").add_variable("th5").add_variable("th6").add_variable("th7").add_variable("th8").
    add_variable("th9").add_variable("th10").add_variable("th11").add_variable("th12").add_variable("th13").
    add_variable("th14").add_variable("th15").add_variable("th16").add_variable("th17").add_variable("th18").
    add_variable("th21").add_variable("th24").add_variable("th27").add_variable("th30").add_variable("th33").
    add_variable("th36").add_variable("th39").add_variable("th42").add_variable("th45").add_variable("th48").
    add_variable("th54").add_variable("th60").add_variable("th66").add_variable("th72").add_variable("th78").
    add_variable("th84").add_variable("th90").add_variable("th96").add_variable("th102").add_variable("th108").
    add_variable("th114"),
    lon=lonp1,
    lat=latp1,
    # size=100,                # 粒子总数
    depth=-2*np.ones(lonp1.shape),
    time=timep2,
    # repeatdt=repeatdt,
    # chunks=(10000, 1),
)


output_file = pset.ParticleFile(
    name=outputwavename, outputdt=timedelta(hours=1)
)

pset.execute(
    [SampleT,parcels.AdvectionRK4,CheckOutOfBounds],  # simply combine the Kernels in a list
    runtime=timedelta(hours=2148),
    dt=timedelta(seconds=600),
    output_file=output_file,
)

# pset.repeatdt = None


# pset.execute(
#     [SampleT,parcels.AdvectionRK4,CheckOutOfBounds],  # simply combine the Kernels in a list
#     # runtime=timedelta(hours=9),
#     runtime=timedelta(hours=2098),
#     dt=timedelta(seconds=600),
#     output_file=output_file,
# )

/home/simingzhang/anaconda3/lib/python3.12/site-packages/parcels/field.py:1180: RuntimeWarning: Sampling of velocities should normally be done using fieldset.UV or fieldset.UVW object; tread carefully
  self._check_velocitysampling()


INFO: Output files are stored in /meddy/simingzhang/Data/Parcels_data/wave_pars_P2500T89.5days_rot.zarr.
  4%|███▊                                                                                           | 313216.0/7732800.0 [02:39<1:05:16, 1894.56it/s]

In [ ]:
# outputwavename=f'{wave_dir}wave_pars_P{100000}T{89.5}days.zarr'

In [ ]:
outputwavename

In [ ]:
# zarr_path = '/meddy/simingzhang/Data/Parcels_data/wave_pars_P103462.0T89.5days.zarr'

# # 使用xarray的open_mfdataset打开分布式存储
# ds = xr.open_mfdataset(
#     f'{zarr_path}/proc*.zarr',  # 匹配所有进程目录
#     engine='zarr',
#     combine='nested',
#     concat_dim='particle',
#     parallel=True
# )


# from glob import glob
# from os import path

# files = glob(path.join(f'{outputwavename}/', "proc*"))
# ds = xr.concat(
#     [xr.open_zarr(f) for f in files],
#     dim="trajectory",
#     compat="no_conflicts",
#     coords="minimal",
# )

In [ ]:
ds = xr.open_zarr(outputwavename)


In [ ]:
ds

In [ ]:
# plt.plot(ds.lon.T, ds.lat.T, ".-")
# plt.xlabel("lon")
# plt.ylabel("lat")
# plt.show()

# transform velocity

In [ ]:
ue,ve=transunit_spher2flat(ds['ue'].values,ds['ve'].values,ds['lat'].values)
ds['ue'].values=ue
ds['ve'].values=ve

In [ ]:
f'{outputwavename[37:-5]}.nc'

In [ ]:
ds.to_netcdf(f'{outputwavename[:-5]}.nc')

In [ ]:
# !jupyter nbconvert --to python Parcels_Iceland_wave.ipynb


In [ ]:
# (ds['time'].values/1e9/3600)[:,0].astype(int)

In [ ]:
# list(ds.data_vars)[10]

In [ ]:
# ds['lon'].values.shape[0]

In [ ]:
# LONt=np.zeros((ds['lon'].values.shape))
# for ii in range(ds['lon'].values.shape[0]):
#     TT=ds['time'].values[ii,:].astype(int)
#     indice=int(TT[0]/3600/1e9)
#     midvar=np.zeros(TT.shape)*np.nan;
#     midvar[0+indice:]=(ds['lon'].values)[ii,0:(TT.shape[0]-indice)];
#     LONt[ii,:]=midvar;
#     print(ii)
# ds['lon'].values=LONt

In [26]:
# ds[list(ds.data_vars)[0]]

In [27]:
# ds1=ds

In [28]:
# ds1